In [9]:
import os
import gc
import time
import random
import pickle
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import WeightedRandomSampler

# Torchvision
from torchvision import models
from torchvision import transforms

# Metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

# Visualization
import matplotlib.pyplot as plt

print("Imports successful.")

Imports successful.


In [10]:
# DEVICE SETUP

torch.set_num_threads(os.cpu_count())

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("=" * 50)
    print("GPU DETECTED")
    print("=" * 50)

    print("GPU Name:")
    print(torch.cuda.get_device_name(0))

    print("\nCUDA Version:")
    print(torch.version.cuda)

else:

    device = torch.device("cpu")

    print("=" * 50)
    print("NO CUDA GPU DETECTED")
    print("=" * 50)

    print("Using CPU.")

NO CUDA GPU DETECTED
Using CPU.


In [11]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds initialized.")

Seeds initialized.


In [ ]:
# =========================================================
# CELL 4 — DATASET PATHS
# LOCAL OR KAGGLE
# =========================================================

USE_KAGGLE = False

# =========================================================
# LOCAL PATHS
# =========================================================

LOCAL_DATA_DIR = "D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed"

# =========================================================
# KAGGLE PATHS
# CHANGE DATASET NAME LATER
# =========================================================

KAGGLE_DATA_DIR = "/kaggle/input/YOUR_DATASET_NAME/processed"

# =========================================================
# SELECT PATH
# =========================================================

if USE_KAGGLE:

    DATA_DIR = KAGGLE_DATA_DIR

else:

    DATA_DIR = LOCAL_DATA_DIR

# =========================================================
# PKL FILES
# =========================================================

TRAIN_PKL = os.path.join(DATA_DIR, "autsl_train.pkl")

VAL_PKL = os.path.join(DATA_DIR, "autsl_val.pkl")

TEST_PKL = os.path.join(DATA_DIR, "autsl_test.pkl")

print(TRAIN_PKL)
print(VAL_PKL)
print(TEST_PKL)

D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_train.pkl
D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_val.pkl
D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed\wlasl_test.pkl


In [13]:
# CONFIG
CONFIG = {

    "batch_size": 8 if torch.cuda.is_available() else 2,

    "epochs": 3,

    "learning_rate": 1e-4,

    "weight_decay": 1e-4,

    "dropout": 0.3,

    "hidden_size": 128,

    "lstm_layers": 2,

    "gradient_clip": 1.0,

    "early_stopping_patience": 5,

    "scheduler_patience": 2,

    "subset_size": 500,

    "freeze_cnn": True,

    "num_workers": 2 if torch.cuda.is_available() else 0
}

print(CONFIG)

{'batch_size': 2, 'epochs': 3, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'dropout': 0.3, 'hidden_size': 128, 'lstm_layers': 2, 'gradient_clip': 1.0, 'early_stopping_patience': 5, 'scheduler_patience': 2, 'subset_size': 500, 'freeze_cnn': True, 'num_workers': 0}


In [14]:
# =========================================================
# CELL 6 — AUGMENTATION
# =========================================================

class VideoAugmentation:

    def __init__(self):

        self.transform = transforms.Compose([

            transforms.ToPILImage(),

            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2
            ),

            transforms.RandomRotation(5),

            transforms.ToTensor()
        ])

    def __call__(self, frames):

        augmented_frames = []

        for frame in frames:

            frame = self.transform(frame)

            augmented_frames.append(frame)

        return torch.stack(augmented_frames)

In [ ]:
# =========================================================
# CELL 7 — DATASET CLASS
# =========================================================

class SignLanguageDataset(Dataset):

    def __init__(
        self,
        pkl_path,
        train=False,
        subset_size=None
    ):

        with open(pkl_path, "rb") as f:

            data = pickle.load(f)

        self.frames = np.array(self.frames)
        self.labels = np.array(self.labels)

        # =================================================
        # RANDOM SUBSET
        # =================================================

        if subset_size is not None:

            subset_size = min(subset_size, len(self.labels))

            indices = np.random.choice(
                len(self.labels),
                subset_size,
                replace=False
            )

            self.frames = self.frames[indices]
            self.labels = self.labels[indices]

        self.train = train

        self.augment = VideoAugmentation()

        print(f"Loaded {len(self.labels)} samples.")

        print("Unique classes:",
              len(np.unique(self.labels)))

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        video = self.frames[idx]

        # [T,H,W,C] -> [T,C,H,W]

        video = torch.tensor(video).permute(0,3,1,2).float()

        # Normalize

        video = video / 255.0

        # Augmentation

        if self.train:

            video = self.augment(video)

        label = torch.tensor(
            self.labels[idx]
        ).long()

        return video, label

In [ ]:
# =========================================================
# CELL 8 — LOAD DATASETS
# =========================================================

train_dataset = SignLanguageDataset(
    TRAIN_PKL,
    train=True,
    subset_size=CONFIG["subset_size"]
)

val_dataset = SignLanguageDataset(
    VAL_PKL,
    train=False,
    subset_size=200
)

test_dataset = SignLanguageDataset(
    TEST_PKL,
    train=False,
    subset_size=200
)

print(train_dataset.labels[:20])

print(
    "Unique classes:",
    len(np.unique(train_dataset.labels))
)

Loaded 500 samples.
Unique classes: 1
Loaded 200 samples.
Unique classes: 1
Loaded 200 samples.
Unique classes: 1


In [17]:
# =========================================================
# CELL 9 — CLASS BALANCING
# =========================================================

from collections import Counter

class_counts = Counter(train_dataset.labels)

sample_weights = []

for label in train_dataset.labels:

    weight = 1.0 / class_counts[label]

    sample_weights.append(weight)

sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Class balancing initialized.")

Class balancing initialized.


In [18]:
# =========================================================
# CELL 10 — DATALOADERS
# =========================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=CONFIG["batch_size"],

    sampler=sampler,

    num_workers=CONFIG["num_workers"]
)

val_loader = DataLoader(

    val_dataset,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=CONFIG["num_workers"]
)

test_loader = DataLoader(

    test_dataset,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=CONFIG["num_workers"]
)

print("DataLoaders ready.")

DataLoaders ready.


In [19]:
# =========================================================
# CELL 11 — MODEL
# =========================================================

class ResNet18_BiLSTM(nn.Module):

    def __init__(
        self,
        num_classes,
        hidden_size=128,
        lstm_layers=2,
        dropout=0.3
    ):

        super().__init__()

        # =================================================
        # RESNET18
        # =================================================

        resnet = models.resnet18(
            weights="DEFAULT"
        )

        self.cnn = nn.Sequential(
            *list(resnet.children())[:-1]
        )

        # =================================================
        # FREEZE CNN
        # =================================================

        if CONFIG["freeze_cnn"]:

            for param in self.cnn.parameters():

                param.requires_grad = False

        self.feature_size = 512

        # =================================================
        # BILSTM
        # =================================================

        self.lstm = nn.LSTM(

            input_size=self.feature_size,

            hidden_size=hidden_size,

            num_layers=lstm_layers,

            batch_first=True,

            bidirectional=True,

            dropout=dropout
        )

        # =================================================
        # DROPOUT
        # =================================================

        self.dropout = nn.Dropout(dropout)

        # =================================================
        # CLASSIFIER
        # =================================================

        self.classifier = nn.Sequential(

            nn.Linear(hidden_size * 2, 128),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        B, T, C, H, W = x.shape

        # =================================================
        # CNN FEATURES
        # =================================================

        x = x.view(B*T, C, H, W)

        features = self.cnn(x)

        features = features.squeeze(-1).squeeze(-1)

        # =================================================
        # RESTORE TEMPORAL DIMENSION
        # =================================================

        features = features.view(
            B,
            T,
            self.feature_size
        )

        # =================================================
        # LSTM
        # =================================================

        lstm_out, _ = self.lstm(features)

        out = lstm_out[:, -1, :]

        out = self.dropout(out)

        out = self.classifier(out)

        return out

In [20]:
# =========================================================
# CELL 12 — INITIALIZE MODEL
# =========================================================

num_classes = len(
    np.unique(train_dataset.labels)
)

model = ResNet18_BiLSTM(

    num_classes=num_classes,

    hidden_size=CONFIG["hidden_size"],

    lstm_layers=CONFIG["lstm_layers"],

    dropout=CONFIG["dropout"]
)

model = model.to(device)

print(model)

ResNet18_BiLSTM(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=T

In [21]:
# =========================================================
# CELL 13 — LOSS + OPTIMIZER + SCHEDULER
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(

    model.parameters(),

    lr=CONFIG["learning_rate"],

    weight_decay=CONFIG["weight_decay"]
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=CONFIG["scheduler_patience"]
)

print("Training components initialized.")

Training components initialized.


In [22]:
# =========================================================
# CELL 14 — TRAIN FUNCTION
# =========================================================

def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    all_preds = []
    all_labels = []

    for videos, labels in loader:

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()

        # =============================================
        # GRADIENT CLIPPING
        # =============================================

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["gradient_clip"]
        )

        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    return avg_loss, accuracy

In [23]:
# =========================================================
# CELL 15 — VALIDATION FUNCTION
# =========================================================

def validate(model, loader):

    model.eval()

    running_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for videos, labels in loader:

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = running_loss / len(loader)

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    return avg_loss, accuracy, all_preds, all_labels

In [24]:
# =========================================================
# CELL 16 — TRAINING LOOP
# =========================================================

best_val_loss = float("inf")

early_stop_counter = 0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

start_time = time.time()

for epoch in range(CONFIG["epochs"]):

    print("\n" + "="*50)
    print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
    print("="*50)

    # =====================================================
    # TRAIN
    # =====================================================

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    # =====================================================
    # VALIDATION
    # =====================================================

    val_loss, val_acc, _, _ = validate(
        model,
        val_loader
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")

    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")

    # =====================================================
    # SAVE BEST MODEL
    # =====================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        early_stop_counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print("Best model saved.")

    else:

        early_stop_counter += 1

        print(
            f"Early stopping counter: "
            f"{early_stop_counter}"
        )

    # =====================================================
    # EARLY STOPPING
    # =====================================================

    if early_stop_counter >= \
       CONFIG["early_stopping_patience"]:

        print("\nEarly stopping triggered.")

        break

total_time = time.time() - start_time

print("\nTraining Complete.")

print(f"\nTotal Training Time:"
      f" {total_time/60:.2f} minutes")


Epoch 1/3
Train Loss: 0.0000
Train Accuracy: 1.0000
Val Loss: 0.0000
Val Accuracy: 1.0000
Best model saved.

Epoch 2/3
Train Loss: 0.0000
Train Accuracy: 1.0000
Val Loss: 0.0000
Val Accuracy: 1.0000
Early stopping counter: 1

Epoch 3/3


KeyboardInterrupt: 

In [ ]:
# =========================================================
# CELL 17 — LOAD BEST MODEL
# =========================================================

model.load_state_dict(
    torch.load(
        "best_model.pth",
        map_location=device
    )
)

print("Best model loaded.")

In [ ]:
# =========================================================
# CELL 18 — TEST EVALUATION
# =========================================================

test_loss, test_acc, \
test_preds, test_labels = validate(

    model,
    test_loader
)

print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Test Loss: {test_loss:.4f}")

print(f"Test Accuracy: {test_acc:.4f}")

print(f"Test F1 Score: "
      f"{f1_score(test_labels, test_preds, average='weighted'):.4f}")

In [ ]:
# =========================================================
# CELL 19 — CLASSIFICATION REPORT
# =========================================================

report = classification_report(
    test_labels,
    test_preds
)

print(report)

In [ ]:
# =========================================================
# CELL 20 — CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    test_labels,
    test_preds
)

print(cm)

In [ ]:
# =========================================================
# CELL 21 — PLOTS
# =========================================================

plt.figure(figsize=(10,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Loss Curve")

plt.legend()

plt.show()

In [ ]:
# =========================================================
# CELL 22 — ACCURACY PLOT
# =========================================================

plt.figure(figsize=(10,5))

plt.plot(train_accuracies,
         label="Train Accuracy")

plt.plot(val_accuracies,
         label="Validation Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Accuracy Curve")

plt.legend()

plt.show()

In [ ]:
# =========================================================
# CELL 23 — INFERENCE FUNCTION
# =========================================================

def predict_video(model, video_tensor):

    model.eval()

    with torch.no_grad():

        video_tensor = video_tensor.unsqueeze(0)

        video_tensor = video_tensor.to(device)

        outputs = model(video_tensor)

        probs = torch.softmax(outputs, dim=1)

        pred_class = torch.argmax(
            probs,
            dim=1
        ).item()

    return pred_class

print("Inference function ready.")

In [ ]:
# =========================================================
# CELL 24 — SAVE METRICS
# =========================================================

metrics_df = pd.DataFrame({

    "train_loss": train_losses,

    "val_loss": val_losses,

    "train_accuracy": train_accuracies,

    "val_accuracy": val_accuracies
})

metrics_df.to_csv(
    "training_metrics.csv",
    index=False
)

print("Metrics saved.")